In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gradio as gr

In [ ]:
# 1. Setup and Data Loading
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_PATH, IMG_DIR = "socal2.csv", "socal2/socal_pics"
df = pd.read_csv(CSV_PATH)
df['image_path'] = df['image_id'].apply(lambda x: os.path.join(IMG_DIR, f"{x}.jpg"))
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)


In [ ]:
# 2. Preprocessing
num_cols = ['bed', 'bath', 'sqft']
X_tab, y, imgs = df[num_cols], df['price'].values.astype(np.float32), df['image_path'].values
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(X_tab, y, imgs, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

img_trans = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class HousingDataset(Dataset):
    def __init__(self, tab, paths, targets):
        self.tab, self.paths, self.targets = tab, paths, targets
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = img_trans(Image.open(self.paths[i]).convert('RGB'))
        return img, torch.tensor(self.tab[i], dtype=torch.float32), torch.tensor(self.targets[i])

train_loader = DataLoader(HousingDataset(X_train_s, img_train, y_train), batch_size=32, shuffle=True)


In [ ]:
# 3. Model Definition
class MultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Identity()
        self.mlp = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fuse = nn.Sequential(nn.Linear(512 + 32, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, img, tab):
        return self.fuse(torch.cat((self.cnn(img), self.mlp(tab)), dim=1)).squeeze()

model = MultimodalModel().to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
crit = nn.MSELoss()

In [ ]:
# 4. Simple Training Loop (5 Epochs)
for e in range(5):
    model.train()
    for img, tab, target in train_loader:
        img, tab, target = img.to(device), tab.to(device), target.to(device)
        opt.zero_grad()
        loss = crit(model(img, tab), target)
        loss.backward()
        opt.step()
    print(f"Epoch {e+1} Complete")


In [ ]:
# 5. Gradio Interface
def predict(image, bed, bath, sqft):
    model.eval()
    img_t = img_trans(image).unsqueeze(0).to(device)
    tab_t = torch.tensor(scaler.transform([[bed, bath, sqft]]), dtype=torch.float32).to(device)
    with torch.no_grad():
        pred = model(img_t, tab_t).item()
    return f"Predicted Price: ${pred:,.2f}"

gr.Interface(fn=predict, inputs=[gr.Image(type="pil"), "number", "number", "number"], 
             outputs="text", title="Housing Price Predictor").launch()